# B2 — Prototype: Semantic Search on 1,000-Row Sample

**Goal:** Verify that the full pipeline (embed → FAISS → search) works *before* Akriti's corpus arrives.  
Uses a 1,000-row random sample from the raw EMSCAD CSV.

> See task plan § B2 for instructions.

In [ ]:
import time, numpy as np, pandas as pd, faiss
from sentence_transformers import SentenceTransformer

# ── Config ──────────────────────────────────────────────────────────────────
CSV_PATH   = "../data/raw/fake_job_postings.csv"   # download from Kaggle first
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
SAMPLE_N   = 1000
SEED       = 42
# ────────────────────────────────────────────────────────────────────────────

df_raw = pd.read_csv(CSV_PATH)
df     = df_raw.sample(SAMPLE_N, random_state=SEED).reset_index(drop=True)
print(f"Sample size: {len(df)}  |  Columns: {df.columns.tolist()}")

In [ ]:
# Build a quick search_text: title + description + requirements
def make_text(row):
    parts = [str(row.get('title','')), str(row.get('description','')), str(row.get('requirements',''))]
    return ' '.join(p for p in parts if p and p.lower() != 'nan')

df['search_text'] = df.apply(make_text, axis=1)
doc_ids = [f"sample_{i}" for i in range(len(df))]
texts   = df['search_text'].tolist()

print("Median text length (words):", pd.Series(texts).str.split().apply(len).median())

In [ ]:
# ── Embed ───────────────────────────────────────────────────────────────────
model = SentenceTransformer(MODEL_NAME)
print(f"max_seq_length = {model.max_seq_length}")

t0 = time.time()
emb = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
emb = np.asarray(emb, dtype='float32')
elapsed = time.time() - t0
print(f"Embedded {len(texts)} texts in {elapsed:.1f}s  → shape {emb.shape}")
print(f"Estimated time for 18k postings: {elapsed/SAMPLE_N*18000/60:.1f} min")

In [ ]:
# ── Build FAISS index ────────────────────────────────────────────────────────
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
print(f"Index size: {index.ntotal} vectors")

In [ ]:
# ── Search 3 test queries ───────────────────────────────────────────────────
test_queries = [
    "entry-level data analyst internship requiring Python and SQL",
    "machine learning internship for a fresher with Python",
    "work-from-home data entry role promising high earnings",
]

for query in test_queries:
    q_vec = model.encode([query], normalize_embeddings=True).astype('float32')
    scores, idxs = index.search(q_vec, 5)
    print(f"\n🔍 Query: {query}")
    for rank, (i, s) in enumerate(zip(idxs[0], scores[0]), 1):
        title = df.iloc[i]['title']
        print(f"  {rank}. [{doc_ids[i]}] {title[:80]}  (score={s:.4f})")

## Observations

_Fill these in after running the cells above (B2 step 3):_

1. **Query 1 (data analyst, Python/SQL):** Top results look … 
2. **Query 2 (ML fresher):** …
3. **Query 3 (risky WFH):** …

**Timing estimate:** ~___ min for 18k postings on this machine.